In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import joblib
from sklearn import metrics
from sklearn.metrics import confusion_matrix, classification_report

## 1. Load Pre-trained Model

In [ ]:
# Load the trained model
model_path = '../assets/mnist_ml_model.pkl'
model = joblib.load(model_path)

print("=" * 60)
print("✓ Model loaded successfully!")
print("=" * 60)
print(f"Model path: {model_path}")
print(f"Model type: {type(model).__name__}")
print("=" * 60)

## 2. Load Test Data

In [ ]:
def load_data(path):
    """Load .npz file and convert to DataFrame"""
    data_npz = np.load(path)
    
    # Access arrays
    X_images = data_npz['train_images']
    y_labels = data_npz['train_labels']
    
    # Flatten images from 28x28 to 784
    images_flat = X_images.reshape(X_images.shape[0], -1)
    
    # Create DataFrame
    data = pd.DataFrame(images_flat)
    data.insert(0, 'label', y_labels)
    data.index += 1
    
    return data

def split_Xy(data):
    """Split data into features and labels"""
    return data.iloc[:, 1:], data.iloc[:, 0]

# Load augmented font dataset
aug_data = load_data("../assets/mnist_aug.npz")

print(f"Augmented dataset shape: {aug_data.shape}")
print(f"\nLabel distribution:")
print(aug_data['label'].value_counts().sort_index())

## 3. Visualize Sample Images

In [ ]:
def plot_digit_on_ax(row, ax):
    """Helper function to plot digit on a specific axis"""
    image = row[1:].to_numpy().reshape(28, 28)
    ax.imshow(image, cmap="binary")
    ax.axis("off")

# Visualize some samples from augmented dataset
print("Sample augmented images:")
fig, axes = plt.subplots(2, 5, figsize=(12, 5))
axes = axes.flatten()

for i, ax in enumerate(axes):
    sample_idx = np.random.randint(0, len(aug_data))
    plot_digit_on_ax(aug_data.iloc[sample_idx], ax)
    ax.set_title(f"Label: {aug_data.iloc[sample_idx]['label']}")

plt.tight_layout()
plt.show()

## 4. Test Model on Augmented Data

In [ ]:
# Split features and labels
X_aug, y_aug = split_Xy(aug_data)

print("=" * 60)
print("Testing MNIST Model on Augmented Font Data")
print("=" * 60)

# Make predictions
y_aug_pred = model.predict(X_aug)

# Calculate accuracy
aug_accuracy = metrics.accuracy_score(y_aug, y_aug_pred)

print(f"\n📊 Test Results:")
print(f"  Total samples:          {len(y_aug)}")
print(f"  Correctly classified:   {(y_aug == y_aug_pred).sum()}")
print(f"  Misclassified:          {(y_aug != y_aug_pred).sum()}")
print(f"  Accuracy:               {aug_accuracy:.4f} ({aug_accuracy*100:.2f}%)")

print("\n" + "=" * 60)

## 5. Confusion Matrix Analysis

In [ ]:
print("=" * 60)
print("Confusion Matrix Analysis")
print("=" * 60)

# Confusion matrix
cm = confusion_matrix(y_aug, y_aug_pred)

# Plot confusion matrix
fig, ax = plt.subplots(figsize=(10, 8))
im = ax.imshow(cm, cmap='Blues')

# Labels
ax.set_xticks(np.arange(10))
ax.set_yticks(np.arange(10))
ax.set_xlabel('Predicted Label', fontsize=12)
ax.set_ylabel('True Label', fontsize=12)
ax.set_title('Confusion Matrix: MNIST Model on Augmented Font Data', fontsize=14, fontweight='bold')

# Add text annotations
for i in range(10):
    for j in range(10):
        text = ax.text(j, i, cm[i, j], ha="center", va="center", 
                      color="white" if cm[i, j] > cm.max() / 2 else "black")

plt.colorbar(im, ax=ax)
plt.tight_layout()
plt.show()

print("\n📋 Classification Report:")
print(classification_report(y_aug, y_aug_pred, target_names=[str(i) for i in range(10)]))

## 6. Analyze Misclassified Samples

In [ ]:
print("=" * 60)
print("Most Confused Predictions")
print("=" * 60)

# Find misclassified samples
misclassified = (y_aug != y_aug_pred)
misclassified_indices = np.where(misclassified)[0]

print(f"\nTotal misclassified: {len(misclassified_indices)} / {len(y_aug)} ({len(misclassified_indices)/len(y_aug)*100:.2f}%)")

# Show some misclassified examples
if len(misclassified_indices) > 0:
    num_examples = min(10, len(misclassified_indices))
    sample_indices = np.random.choice(misclassified_indices, num_examples, replace=False)
    
    fig, axes = plt.subplots(2, 5, figsize=(15, 6))
    axes = axes.flatten()
    
    for idx, sample_idx in enumerate(sample_indices):
        ax = axes[idx]
        image = X_aug.iloc[sample_idx].to_numpy().reshape(28, 28)
        ax.imshow(image, cmap='binary')
        ax.axis('off')
        true_label = y_aug.iloc[sample_idx]
        pred_label = y_aug_pred[sample_idx]
        ax.set_title(f'True: {true_label}\nPred: {pred_label}', 
                    color='red', fontweight='bold')
    
    plt.suptitle('Misclassified Augmented Font Samples', fontsize=16, fontweight='bold')
    plt.tight_layout()
    plt.show()
else:
    print("No misclassified samples! Perfect accuracy!")

## 7. Per-Digit Analysis

In [ ]:
# Analyze accuracy per digit
print("=" * 60)
print("Per-Digit Accuracy Analysis")
print("=" * 60)

for digit in range(10):
    digit_mask = (y_aug == digit)
    digit_total = digit_mask.sum()
    digit_correct = ((y_aug == digit) & (y_aug_pred == digit)).sum()
    digit_accuracy = digit_correct / digit_total if digit_total > 0 else 0
    
    print(f"Digit {digit}: {digit_correct:4d}/{digit_total:4d} = {digit_accuracy:.4f} ({digit_accuracy*100:.2f}%)")

print("=" * 60)

## Summary

This analysis shows how a model trained on **standard handwritten MNIST digits** performs when tested on **synthetic font-based augmented data** with various transformations (rotation, blur, position offset, font size/style variations).

**Key Insights:**
- **Accuracy** indicates overall model robustness to font-based variations
- **Confusion matrix** reveals which digit pairs are most commonly confused
- **Misclassified examples** show specific cases where font-based features confuse the model
- **Per-digit accuracy** identifies which digits are most challenging with augmentation

This is useful for understanding:
1. The model's robustness to font variations
2. Which transformations cause the most confusion
3. Whether additional training with augmented data would improve generalization
4. Which digits need more attention in future training